In [ ]:
from catboost import CatBoostRegressor
import mlflow
import numpy as np
import optuna
from optuna_integration.catboost import CatBoostPruningCallback
import pandas as pd

from restaurant_visitor_eda.config import PROCESSED_DATA_DIR
from restaurant_visitor_eda.features import (
    binary_features,
    categorical_features,
    get_custom_cv_splits,
    numeric_features,
)

In [ ]:
def add_store_age_features(
    df_train: pd.DataFrame, df_test: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    first_open_dates = df_train.groupby("air_store_id")["visit_date"].min().reset_index()
    first_open_dates.columns = ["air_store_id", "first_open_date"]

    df_train = pd.merge(df_train, first_open_dates, on="air_store_id", how="left")
    df_test = pd.merge(df_test, first_open_dates, on="air_store_id", how="left")

    df_train["days_since_first_open"] = (
        df_train["visit_date"] - df_train["first_open_date"]
    ).dt.days
    df_test["days_since_first_open"] = (df_test["visit_date"] - df_test["first_open_date"]).dt.days

    df_train.drop(columns=["first_open_date"], inplace=True)
    df_test.drop(columns=["first_open_date"], inplace=True)

    return df_train, df_test


def compute_calendar_lags_hierarchical(df: pd.DataFrame, lags: list[int]) -> pd.DataFrame:
    res_df = df.copy()

    for lag in lags:
        shifted = df[["air_store_id", "visit_date", "visitors"]].copy()
        shifted["visit_date"] = shifted["visit_date"] + pd.to_timedelta(lag, unit="D")
        shifted = shifted.rename(columns={"visitors": f"lag_{lag}"})

        res_df = pd.merge(res_df, shifted, on=["air_store_id", "visit_date"], how="left")

        res_df[f"lag_{lag}"] = (
            res_df[f"lag_{lag}"]
            .fillna(res_df["store_dow_mean_cum"])
            .fillna(res_df["store_mean_cum"])
            .fillna(res_df["genre_geo_mean_cum"])
        )

    return res_df

In [ ]:
df_train = pd.read_csv(PROCESSED_DATA_DIR / "train_features.csv", parse_dates=["visit_date"])
df_test = pd.read_csv(PROCESSED_DATA_DIR / "test_features.csv", parse_dates=["visit_date"])

print(f"Base Train shape: {df_train.shape}")
print(f"Base Test shape: {df_test.shape}")

df_train, df_test = add_store_age_features(df_train, df_test)

df_train["is_test"] = 0
df_test["is_test"] = 1

df_all = pd.concat([df_train, df_test], ignore_index=True)
df_all = df_all.sort_values(["air_store_id", "visit_date"]).reset_index(drop=True)

df_all = compute_calendar_lags_hierarchical(df_all, lags=[39, 42])

df_train_final = df_all[df_all["is_test"] == 0].copy().reset_index(drop=True)
df_test_final = df_all[df_all["is_test"] == 1].copy().reset_index(drop=True)

df_train_final.drop(columns=["is_test"], inplace=True)
df_test_final.drop(columns=["is_test", "visitors"], inplace=True)

print(f"Engineered Train shape: {df_train_final.shape}")
print(f"Engineered Test shape: {df_test_final.shape}")

In [ ]:
local_categorical_features = categorical_features.copy()
local_binary_features = binary_features.copy()

local_numeric_features = numeric_features.copy()
local_numeric_features.extend(["days_since_first_open", "lag_39", "lag_42"])

features = local_categorical_features + local_numeric_features + local_binary_features

X_full = df_train_final[features]
y_full = np.log1p(df_train_final["visitors"].values)

cv_splits = get_custom_cv_splits(df_train_final, n_splits=3, val_days=39)

In [ ]:
def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 9, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 25.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": 42,
        "od_type": "Iter",
        "od_wait": 50,
        "verbose": False,
    }

    pruning_callback = CatBoostPruningCallback(trial, "RMSE")
    cv_scores = []
    fold_iters = []

    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        X_train, y_train = X_full.iloc[train_idx], y_full[train_idx]
        X_val, y_val = X_full.iloc[val_idx], y_full[val_idx]

        model = CatBoostRegressor(**params, cat_features=local_categorical_features)

        if fold == 0:
            model.fit(X_train, y_train, eval_set=(X_val, y_val), callbacks=[pruning_callback])
            pruning_callback.check_pruned()
        else:
            model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)

        best_score = model.get_best_score()["validation"]["RMSE"]
        best_iter = model.get_best_iteration()

        cv_scores.append(best_score)
        fold_iters.append(best_iter)

    trial.set_user_attr("mean_best_iter", int(np.mean(fold_iters)))

    return np.mean(cv_scores)

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow_tracking.db")
mlflow.set_experiment("CatBoost_Optuna_Tuning_With_Lags")

with mlflow.start_run(run_name="optuna_search_cold_start_mitigation"):
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=50)
    study = optuna.create_study(direction="minimize", pruner=pruner)

    study.optimize(objective, n_trials=50, show_progress_bar=True)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_cv_rmse", study.best_value)
    mlflow.log_metric("mean_best_iter", study.best_trial.user_attrs["mean_best_iter"])

    print(f"\n Best RMSE with lags and age: {study.best_value:.4f}")

In [ ]:
print("\n--- BEST PARAMS ---")
for key, value in study.best_params.items():
    print(f"{key}: {value}")
print(f"\n Best RMSLE during CV: {study.best_value:.4f}")

optimal_iterations = study.best_trial.user_attrs["mean_best_iter"]
print(f"Optimal Iterations: {optimal_iterations}")

In [ ]:
final_params = study.best_params.copy()
final_params["iterations"] = int(optimal_iterations * 1.5)
final_params["loss_function"] = "RMSE"
final_params["eval_metric"] = "RMSE"
final_params["random_seed"] = 42
final_params["learning_rate"] = final_params["learning_rate"] / 1.5

final_model = CatBoostRegressor(**final_params, cat_features=local_categorical_features)
final_model.fit(X_full, y_full, verbose=100)

In [ ]:
X_test = df_test_final[features]

preds_log = final_model.predict(X_test)
preds_real_clipped = np.clip(np.expm1(preds_log), 1.0, None)

submission = pd.DataFrame(
    {
        "id": df_test_final["air_store_id"]
        + "_"
        + df_test_final["visit_date"].dt.strftime("%Y-%m-%d"),
        "visitors": preds_real_clipped,
    }
)

submission_path = "submission_catboost_optuna_lags_and_age.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")
submission.head()

In [ ]:
train_idx, val_idx = cv_splits[-1]

X_train, y_train = X_full.iloc[train_idx], y_full[train_idx]
X_val, y_val = X_full.iloc[val_idx], y_full[val_idx]

model = CatBoostRegressor(
    **study.best_params, cat_features=local_categorical_features, verbose=False
)
model.fit(X_train, y_train)

val_preds_log = model.predict(X_val)

val_df = df_train_final.iloc[val_idx].copy()
val_df["actual_visitors"] = np.expm1(y_val)
val_df["pred_visitors"] = np.expm1(val_preds_log)

val_df["error"] = val_df["pred_visitors"] - val_df["actual_visitors"]
val_df["abs_error"] = val_df["error"].abs()
val_df["squared_log_error"] = (
    np.log1p(val_df["pred_visitors"]) - np.log1p(val_df["actual_visitors"])
) ** 2

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 8))
sns.scatterplot(data=val_df, x="actual_visitors", y="pred_visitors", alpha=0.4, color="teal")

max_val = int(max(val_df["actual_visitors"].max(), val_df["pred_visitors"].max()))
plt.plot([0, max_val], [0, max_val], color="red", linestyle="--", label="Идеальный прогноз")

plt.title("Actual vs predicted")
plt.xlabel("Actual visitors")
plt.ylabel("Predicted visitors")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
daily_errors = (
    val_df.groupby("visit_date")["squared_log_error"].mean().apply(np.sqrt).reset_index()
)
daily_errors.rename(columns={"squared_log_error": "RMSLE"}, inplace=True)

plt.figure(figsize=(14, 5))
plt.plot(
    daily_errors["visit_date"], daily_errors["RMSLE"], marker="o", color="crimson", linewidth=2
)

plt.title("RMSLE by date")
plt.xlabel("Date")
plt.ylabel("RMSLE")
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.show()

In [ ]:
genre_errors = (
    val_df.groupby("air_genre_name")["squared_log_error"].mean().apply(np.sqrt).reset_index()
)
genre_errors.rename(columns={"squared_log_error": "RMSLE"}, inplace=True)
genre_errors = genre_errors.sort_values("RMSLE", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=genre_errors, x="RMSLE", y="air_genre_name", palette="coolwarm")
plt.title("Error by genre")
plt.xlabel("RMSLE")
plt.ylabel("Genre")
plt.grid(True, alpha=0.2)
plt.show()